In [1]:
# ─── Imports ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for report generation
import seaborn as sns
import base64
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

In [2]:
# ─── Load Data ─────────────────────────────────────────────
rfm = pd.read_csv('../data/processed/rfm_clustered.csv')
df  = pd.read_csv('../data/processed/clean_retail.csv',
                  parse_dates=['InvoiceDate'])

print(f"RFM shape: {rfm.shape}")
print(f"Clean data shape: {df.shape}")

RFM shape: (5878, 12)
Clean data shape: (779425, 9)


In [3]:
# ─── Helper Functions ──────────────────────────────────────
# We embed images directly into HTML using base64 encoding
# WHY: Makes the report a single self-contained HTML file
# No external image dependencies — works offline, easy to share

def img_to_base64(fig):
    """Convert matplotlib figure to base64 string for HTML embedding."""
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return img_str

def load_img_base64(path):
    """Load existing image file as base64."""
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [4]:
# ─── Generate Report Charts ────────────────────────────────

# Chart 1: Segment distribution
fig, ax = plt.subplots(figsize=(10, 5))
seg_data = rfm['Segment'].value_counts()
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(seg_data)))
bars = ax.barh(seg_data.index, seg_data.values, color=colors)
for bar, val in zip(bars, seg_data.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_title('Customer Count by Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Customers')
plt.tight_layout()
chart1 = img_to_base64(fig)
print("Chart 1 done")

# Chart 2: Revenue by segment
fig, ax = plt.subplots(figsize=(10, 5))
rev_data = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=True)
colors2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(rev_data)))
bars = ax.barh(rev_data.index, rev_data.values, color=colors2)
for bar, val in zip(bars, rev_data.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'£{val/1000:.0f}K', va='center', fontsize=10)
ax.set_title('Total Revenue by Segment', fontsize=13, fontweight='bold')
ax.set_xlabel('Revenue (GBP)')
plt.tight_layout()
chart2 = img_to_base64(fig)
print("Chart 2 done")

# Chart 3: RFM Score distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rfm['RFM_Score'], bins=30, color='steelblue', 
        edgecolor='white', alpha=0.85)
ax.set_title('RFM Score Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('RFM Score')
ax.set_ylabel('Number of Customers')
ax.axvline(rfm['RFM_Score'].mean(), color='red', linestyle='--',
           label=f"Mean: {rfm['RFM_Score'].mean():.2f}")
ax.legend()
plt.tight_layout()
chart3 = img_to_base64(fig)
print("Chart 3 done")

# Chart 4: Cluster distribution pie
fig, ax = plt.subplots(figsize=(8, 6))
cluster_data = rfm['Cluster_Label'].value_counts()
colors3 = ['#2ECC71', '#3498DB', '#E74C3C', '#F39C12']
ax.pie(cluster_data.values, labels=cluster_data.index,
       autopct='%1.1f%%', colors=colors3[:len(cluster_data)],
       startangle=90)
ax.set_title('ML Cluster Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
chart4 = img_to_base64(fig)
print("Chart 4 done")

print("\nAll charts generated!")

Chart 1 done
Chart 2 done
Chart 3 done
Chart 4 done

All charts generated!


In [5]:
# ─── Segment Profile Table ─────────────────────────────────
profile = rfm.groupby('Segment').agg(
    Customers     = ('Customer ID', 'count'),
    Avg_Recency   = ('Recency', 'mean'),
    Avg_Frequency = ('Frequency', 'mean'),
    Avg_Monetary  = ('Monetary', 'mean'),
    Total_Revenue = ('Monetary', 'sum')
).round(1).sort_values('Total_Revenue', ascending=False)

profile['Revenue_Share%'] = (
    profile['Total_Revenue'] / profile['Total_Revenue'].sum() * 100
).round(1)

# Convert to HTML table
table_html = profile.to_html(classes='data-table', border=0)
print("Profile table built!")
print(profile)

Profile table built!
                     Customers  Avg_Recency  Avg_Frequency  Avg_Monetary  \
Segment                                                                    
Champions                 1299         20.0           17.1        8177.5   
Loyal Customers           1139         73.2            5.9        2080.5   
At Risk                    620        359.8            5.6        2021.1   
Need Attention             575        413.4            2.1         599.7   
Hibernating                704        277.4            1.3         370.4   
Recent Customers           443         28.1            1.5         483.4   
Potential Loyalists        358         80.9            2.7         569.7   
Lost                       645        567.7            1.1         197.6   
Promising                   95        110.6            1.0         152.2   

                     Total_Revenue  Revenue_Share%  
Segment                                             
Champions               10622564.9  

In [6]:
# ─── Generate HTML Report ──────────────────────────────────
# A self-contained HTML report with embedded charts and tables
# Professional, shareable, no dependencies needed to view

report_date = datetime.now().strftime("%B %d, %Y")
total_revenue = rfm['Monetary'].sum()
total_customers = len(rfm)
champion_pct = len(rfm[rfm['Segment']=='Champions']) / total_customers * 100
at_risk_revenue = rfm[rfm['Segment']=='At Risk']['Monetary'].sum()

html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RFM Customer Segmentation Report</title>
<style>
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{ font-family: 'Segoe UI', sans-serif; background: #f5f6fa; color: #2c3e50; }}
  .header {{ background: linear-gradient(135deg, #2c3e50, #3498db); 
             color: white; padding: 40px; text-align: center; }}
  .header h1 {{ font-size: 2.2em; margin-bottom: 8px; }}
  .header p  {{ font-size: 1em; opacity: 0.85; }}
  .container {{ max-width: 1100px; margin: 30px auto; padding: 0 20px; }}
  .kpi-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); 
               gap: 16px; margin: 30px 0; }}
  .kpi-card {{ background: white; border-radius: 12px; padding: 20px;
               text-align: center; box-shadow: 0 2px 8px rgba(0,0,0,0.08);
               border-top: 4px solid #3498db; }}
  .kpi-value {{ font-size: 1.8em; font-weight: 700; color: #2c3e50; }}
  .kpi-label {{ font-size: 0.85em; color: #7f8c8d; margin-top: 5px; }}
  .section {{ background: white; border-radius: 12px; padding: 25px;
              margin: 20px 0; box-shadow: 0 2px 8px rgba(0,0,0,0.08); }}
  .section h2 {{ font-size: 1.3em; color: #2c3e50; margin-bottom: 15px;
                 padding-bottom: 10px; border-bottom: 2px solid #ecf0f1; }}
  .chart-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  .chart-box img {{ width: 100%; border-radius: 8px; }}
  .data-table {{ width: 100%; border-collapse: collapse; font-size: 0.9em; }}
  .data-table th {{ background: #2c3e50; color: white; padding: 10px 12px;
                    text-align: left; }}
  .data-table td {{ padding: 9px 12px; border-bottom: 1px solid #ecf0f1; }}
  .data-table tr:hover {{ background: #f8f9fa; }}
  .rec-grid {{ display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; }}
  .rec-card {{ background: #f8f9fa; border-radius: 10px; padding: 18px;
               border-left: 4px solid #3498db; }}
  .rec-card h3 {{ color: #2c3e50; margin-bottom: 8px; }}
  .rec-card p  {{ font-size: 0.88em; color: #555; line-height: 1.6; }}
  .tag {{ display: inline-block; background: #3498db; color: white;
          padding: 2px 10px; border-radius: 20px; font-size: 0.78em;
          margin-bottom: 6px; }}
  .footer {{ text-align: center; padding: 20px; color: #95a5a6; font-size: 0.85em; }}
</style>
</head>
<body>

<div class="header">
  <h1>📊 RFM Customer Segmentation Report</h1>
  <p>Advanced customer analysis using RFM scoring and K-Means clustering</p>
  <p style="margin-top:8px; opacity:0.7;">Generated on {report_date} | UCI Online Retail II Dataset</p>
</div>

<div class="container">

  <!-- KPI Cards -->
  <div class="kpi-grid">
    <div class="kpi-card">
      <div class="kpi-value">{total_customers:,}</div>
      <div class="kpi-label">Total Customers Analyzed</div>
    </div>
    <div class="kpi-card" style="border-top-color:#2ECC71">
      <div class="kpi-value">£{total_revenue:,.0f}</div>
      <div class="kpi-label">Total Revenue (GBP)</div>
    </div>
    <div class="kpi-card" style="border-top-color:#E74C3C">
      <div class="kpi-value">{champion_pct:.1f}%</div>
      <div class="kpi-label">Champion Customers</div>
    </div>
    <div class="kpi-card" style="border-top-color:#F39C12">
      <div class="kpi-value">£{at_risk_revenue:,.0f}</div>
      <div class="kpi-label">At-Risk Revenue</div>
    </div>
  </div>

  <!-- Charts -->
  <div class="section">
    <h2>📈 Segment Analysis</h2>
    <div class="chart-grid">
      <div class="chart-box">
        <img src="data:image/png;base64,{chart1}" alt="Segment Distribution"/>
      </div>
      <div class="chart-box">
        <img src="data:image/png;base64,{chart2}" alt="Revenue by Segment"/>
      </div>
    </div>
  </div>

  <div class="section">
    <h2>🤖 ML Clustering & RFM Scores</h2>
    <div class="chart-grid">
      <div class="chart-box">
        <img src="data:image/png;base64,{chart3}" alt="RFM Score Distribution"/>
      </div>
      <div class="chart-box">
        <img src="data:image/png;base64,{chart4}" alt="Cluster Distribution"/>
      </div>
    </div>
  </div>

  <!-- Segment Profile Table -->
  <div class="section">
    <h2>📋 Segment Profile Summary</h2>
    {table_html}
  </div>

  <!-- Business Recommendations -->
  <div class="section">
    <h2>💡 Business Recommendations</h2>
    <div class="rec-grid">
      <div class="rec-card">
        <span class="tag">Champions</span>
        <h3>Reward & Retain</h3>
        <p>Your best customers. Offer loyalty rewards, early product access, 
        and VIP treatment. Focus on retention — losing one Champion is costly.</p>
      </div>
      <div class="rec-card" style="border-left-color:#E74C3C">
        <span class="tag" style="background:#E74C3C">At Risk</span>
        <h3>Win-Back Campaign</h3>
        <p>Previously active customers showing disengagement. 
        Launch targeted win-back emails with special offers within 30 days.</p>
      </div>
      <div class="rec-card" style="border-left-color:#2ECC71">
        <span class="tag" style="background:#2ECC71">Loyal Customers</span>
        <h3>Upsell & Upgrade</h3>
        <p>Strong base customers. Push them toward Champion status with 
        personalised upsell campaigns and loyalty tier upgrades.</p>
      </div>
      <div class="rec-card" style="border-left-color:#95a5a6">
        <span class="tag" style="background:#95a5a6">Lost</span>
        <h3>Low-Cost Re-engagement</h3>
        <p>Highly inactive customers. Use low-cost email re-engagement only. 
        If no response, sunset from active marketing lists to reduce costs.</p>
      </div>
    </div>
  </div>

</div>

<div class="footer">
  RFM Customer Segmentation Analysis | 
  Built with Python · Pandas · Scikit-learn · Matplotlib |
  Data: UCI Online Retail II
</div>

</body>
</html>
"""

# Save report
output_path = '../reports/rfm_report.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"Report saved to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

Report saved to ../reports/rfm_report.html
File size: 186.8 KB
